# Did the L16 fixes help?

## tl;dr

**Run → Run All Cells.** This notebook reads saved results only: no training, sampling, package installation, job submission, or GPU computation. It works with partial results. Re-run all cells after sampling finishes to refresh the comparison.

First read **Readiness**, then **Population comparison**, **Power spectra**, and **Inspect one map**. Missing results remain *pending*, never zero failures. No fix is declared before its generated maps are inspected.

Training array: `61293788`; automatic sampling array: `61293789`. The automatic sampler waits for **all three** training arms. A completed checkpoint alone is not a completed sampling result.

## Context & Methods

All new runs: L16, 256 training maps, width 768, raw weights, seed 123, 300k nominal optimizer updates, paired initial noise, DPM++ order 2 / 50 steps.

| Condition | Pixel patch size | Initialization | What it tests |
| --- | --- | --- | --- |
| Old L16 fresh300k | 8×8 pixels | Native | Existing baseline; no new training |
| `p8_zero` | 8×8 pixels | Zero conditioning/output projections | Initialization only |
| `p4_native` | 4×4 pixels | Native | Patch size only |
| `p4_zero` | 4×4 pixels | Zero conditioning/output projections | Both changes |

The images remain **128×128 pixels**. Patch 8 gives 256 tokens; patch 4 gives 1024 tokens. Zero-init is applied once before training, not held at zero. Attention/FFN matrices are not zeroed.

### Key Assumptions

Use the corrected **select → z-thin → normalize** training reference. Old samples are re-scored against that reference, not their old power-ratio tables. Pixel cosine is uncentered and uses no spatial alignment. Old matrix sampling used model microbatch 8; new arms use 2. The old 500k continuation and L8 200k are auxiliary controls, not matched fresh300k interventions. One seed and one dataset size cannot prove a universal mechanism.

## Data

### 1. Paths

These defaults are for your Great Lakes account. Keep the existing training root and plan hash. The notebook is self-contained and needs only NumPy, pandas, Matplotlib, PyYAML, and IPython in the Jupyter kernel—not PyTorch or diffusers.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ABLATION_ROOT = Path('/scratch/huterer_root/huterer0/jiamingp/dit_l16_a40_init_patch_v2_datafix')
EXPECTED_PLAN_SHA = 'e814a42ad2026c2b9963a43b017220ff0326800c21d920c4234523e16b06b21f'
INCLUDE_OLD_CONTROLS = True

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})
print('Reading:', ABLATION_ROOT)
print('Kernel:', sys.executable)
print('No compute jobs will be submitted.')

### 2. Readiness and integrity checks

Load only completed, hash-checked sample files with the correct paired noises and provenance. ERROR results are excluded and reported explicitly. Epoch counts below are *nominal scheduled* updates, not verified successful AMP updates. This reads saved logs/receipts, **not live Slurm state**. It does not tensor-load or rehash the multi-GB checkpoints.

In [ ]:
"""Independent NumPy reference matching native cosmodiff's slice-first loader.

The frozen legacy evaluation reader fits center/max over unthinned volumes.
Native load_data first selects volumes, thins z, concatenates the retained
slices, and only then fits log/center/max/tanh. Never replace the native tensor
with this reference or change the native training recipe to match legacy IO.
"""

import hashlib
from pathlib import Path

import numpy as np

CONTRACT = "native_select_then_zthin_then_log_centermax_tanh_v1"


def array_hash(array):
    return hashlib.sha256(np.ascontiguousarray(array).tobytes()).hexdigest()


def source_values(value, count, name):
    if isinstance(value, (list, tuple)):
        if len(value) != count:
            raise ValueError(f"data.{name} must have one value per source")
        return list(value)
    return [value] * count


def load_native_training_reference(config):
    """Read exactly the native retained slices, in native source/volume/z order."""
    data = config["data"]
    if config.get("global", {}).get("dtype", "float32") != "float32":
        raise ValueError("the native-reference contract requires float32")
    if data.get("reshape", "2d") != "2d" or data.get("normalization") != "tanh":
        raise ValueError("the ablation reference requires native 2D/tanh data")
    if data.get("transform") != ["log"] or data.get("log", False):
        raise ValueError("the ablation reference requires the native ['log'] transform")
    if config.get("augmentations"):
        raise ValueError("the ablation reference requires no augmentations")
    paths = data["img_path"]
    paths = list(paths) if isinstance(paths, (list, tuple)) else [paths]
    if not paths:
        raise ValueError("no configured data sources")
    readers = source_values(data.get("img_read_fn"), len(paths), "img_read_fn")
    counts = source_values(data.get("n_samples"), len(paths), "n_samples")
    seeds = source_values(data.get("seed"), len(paths), "seed")
    thins = source_values(data.get("zthin", 1), len(paths), "zthin")
    raw_slices, selections = [], []
    for path, reader, count, seed, thin in zip(paths, readers, counts, seeds, thins):
        if reader != "npy_read_fn" or isinstance(thin, bool) or int(thin) != thin or thin < 1:
            raise ValueError("native reference requires npy_read_fn and positive integer zthin")
        raw = np.load(Path(path), mmap_mode="r", allow_pickle=False)
        if raw.ndim != 4:
            raise ValueError(f"native 2D data requires (volume,z,h,w), found {raw.shape}")
        n = len(raw) if count is None else int(count)
        if isinstance(count, bool) or (count is not None and n != count) or not 0 <= n <= len(raw):
            raise ValueError(f"invalid native n_samples={count}")
        # Native ignores seed when n_samples is None.
        selected = (np.arange(n) if seed is None or count is None
                    else np.random.default_rng(seed).choice(len(raw), size=n, replace=False))
        kept = np.asarray(raw[selected][:, ::int(thin)], dtype=np.float32)
        raw_slices.append(kept.reshape(-1, 1, *raw.shape[-2:]).copy())
        selections.append({"path": str(path), "volume_indices": selected.tolist(),
                           "z_indices": list(range(0, raw.shape[1], int(thin)))})
    slices = np.concatenate(raw_slices, axis=0)
    if not len(slices) or not np.isfinite(slices).all() or np.any(slices <= 0):
        raise ValueError("native log data must contain nonempty finite positive retained slices")
    transformed = np.log(slices)
    kwargs = dict(data.get("norm_kwargs") or {})
    center = kwargs.get("center")
    if center is None:
        # float64 reduction avoids a reference-only reduction-order dependency;
        # the elementwise audit permits only <=2e-6 Torch/NumPy rounding.
        center = float(transformed.mean(dtype=np.float64))
    normalized = transformed - np.float32(center)
    xmax = kwargs.get("xmax")
    if xmax is None:
        xmax = float(np.abs(normalized).max())
    if not np.isfinite(center) or not np.isfinite(xmax) or xmax <= 0:
        raise ValueError("native center/max normalization must be finite and nondegenerate")
    normalized = normalized / np.float32(xmax)
    shifted = normalized - np.float32(kwargs.get("mu", 0.0))
    alpha, beta = float(kwargs.get("alpha", 1.0)), float(kwargs.get("beta", 1.0))
    gamma, delta = float(kwargs.get("gamma", 1.0)), float(kwargs.get("delta", 1.0))
    sigma = float(kwargs.get("sigma", 1.0))
    if not all(np.isfinite(x) and x > 0 for x in (alpha, beta, gamma, delta, sigma)):
        raise ValueError("native tanh parameters must be finite and positive")
    positive = alpha * np.tanh((gamma * shifted) / alpha)
    negative = beta * np.tanh((delta * shifted) / beta)
    reference = np.asarray(np.where(shifted >= 0, positive, negative) * sigma, dtype=np.float32)
    if not np.isfinite(reference).all():
        raise ValueError("native reference contains nonfinite values")
    return reference, {"contract": CONTRACT, "shape": list(reference.shape),
        "dtype": str(reference.dtype), "selected_raw_sha256": array_hash(slices),
        "reference_sha256": array_hash(reference), "center": float(center),
        "xmax": float(xmax), "selections": selections}


def audit_native_dataset(parsed, reference, metadata, torch):
    """Check every element and label, leaving the native dataset untouched."""
    dataset = parsed["data"]
    actual = dataset.arrays.detach().cpu().numpy()
    diagnostic = {"shape": list(actual.shape), "dtype": str(actual.dtype),
                  "expected_shape": list(reference.shape), "expected_dtype": "float32"}
    if actual.shape != reference.shape or actual.dtype != np.float32:
        raise ValueError(f"native training tensor shape/dtype mismatch: {diagnostic}")
    if not np.isfinite(actual).all():
        raise ValueError("native training tensors contain nonfinite values")
    diagnostic["max_abs_delta"] = float(np.max(np.abs(actual - reference)))
    if not np.allclose(actual, reference, rtol=0, atol=2e-6):
        raise ValueError(f"native training tensors disagree with slice-first reference: {diagnostic}")
    labels = dataset.labels
    if (labels is None or labels.dtype != torch.long or tuple(labels.shape) != (len(reference),)
            or bool(torch.any(labels != 0).item())):
        raise ValueError("null class labels must be all-zero long[N]")
    normalization = parsed.get("norm")
    fitted = dict(getattr(normalization, "kwargs", {}))
    for key in ("center", "xmax"):
        value = fitted.get(key)
        if value is None or not np.isfinite(value) or not np.isclose(value, metadata[key], rtol=0, atol=2e-6):
            raise ValueError(f"native fitted {key} disagrees with slice-first reference: {value} vs {metadata[key]}")
    return {"status": "matched", "contract": CONTRACT,
        "training_tensor_sha256": array_hash(actual),
        "training_reference_sha256": array_hash(reference),
        "selected_raw_sha256": metadata["selected_raw_sha256"],
        "reference_max_abs_delta": diagnostic["max_abs_delta"],
        "native_fitted_normalization": {key: float(fitted[key]) for key in ("center", "xmax")},
        "shape": diagnostic["shape"], "dtype": diagnostic["dtype"], "null_labels": "all_zero_long"}


"""Read-only CPU inspection of the frozen L16 A40 experiment; never runs models."""

import hashlib
import json
import re
from pathlib import Path

import numpy as np
import yaml


PLAN_SHA = "e814a42ad2026c2b9963a43b017220ff0326800c21d920c4234523e16b06b21f"
TRAIN_REVISION = "50b685f5999b90a212f696ab38b172dca5d968ab"
MATRIX_SHA = "8404ae7b4f607923251d430da2b78afe07871846269927c0a9e2b6e3d8c0e333"
LABELS = {
    "dit_l16_fresh300k": "Old L16: patch 8, native init (300k)",
    "dit_l16_seed456_500k": "Old L16: patch 8, continued run (500k)",
    "dit_l8_200k": "L8 control: patch 8, native init (200k)",
    "p8_zero": "L16: patch 8 + zero-init (300k)",
    "p4_native": "L16: patch 4 + native init (300k)",
    "p4_zero": "L16: patch 4 + zero-init (300k)",
}
GROUPS = ("near-copy", "intermediate", "low-similarity")
COLORS = {"near-copy": "#31688e", "intermediate": "#a87c20", "low-similarity": "#c05c33"}


def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def array_sha(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text())


def scalar(data, key):
    if key not in data or np.asarray(data[key]).size != 1:
        raise ValueError(f"Missing/non-scalar provenance field: {key}")
    return np.asarray(data[key]).reshape(-1)[0].item()


def fields(images, n=None):
    images = np.asarray(images)
    if (images.ndim != 4 or images.shape[1:] != (1, 128, 128)
            or images.dtype != np.float32 or not np.isfinite(images).all()
            or not len(images) or (n is not None and len(images) != n)):
        raise ValueError(f"Expected finite float32 (N,1,128,128) fields; found {images.shape}/{images.dtype}")
    return images


def nearest_training(images, reference):
    """Uncentered pixel cosine, without alignment/augmentation; also return NN indices."""
    def unit(value):
        flat = value.reshape(len(value), -1).astype(np.float32)
        norm = np.linalg.norm(flat, axis=1, keepdims=True)
        if np.any(norm <= 0):
            raise ValueError("Zero-norm map cannot be assigned a pixel cosine")
        return flat / norm
    target = unit(reference)
    scores, indices = [], []
    for start in range(0, len(images), 32):
        similarity = unit(images[start:start + 32]) @ target.T
        index = similarity.argmax(axis=1)
        scores.extend(similarity[np.arange(len(index)), index])
        indices.extend(index)
    return np.asarray(scores, dtype=np.float32), np.asarray(indices, dtype=int)


def boundary_ratio(images, patch):
    """Mean absolute jump on patch edges / mean absolute jump elsewhere, per image."""
    x = images[:, 0]
    horizontal = np.abs(np.diff(x, axis=2))
    vertical = np.abs(np.diff(x, axis=1))
    columns = (np.arange(127) + 1) % patch == 0
    rows = (np.arange(127) + 1) % patch == 0
    edge_sum = horizontal[:, :, columns].sum(axis=(1, 2), dtype=np.float64)
    edge_sum += vertical[:, rows, :].sum(axis=(1, 2), dtype=np.float64)
    edge_count = int(columns.sum() + rows.sum()) * 128
    inside_sum = horizontal[:, :, ~columns].sum(axis=(1, 2), dtype=np.float64)
    inside_sum += vertical[:, ~rows, :].sum(axis=(1, 2), dtype=np.float64)
    inside_count = int((~columns).sum() + (~rows).sum()) * 128
    return np.divide(edge_sum / edge_count, inside_sum / inside_count,
                     out=np.full(len(x), np.nan), where=inside_sum > 0)


def radial_power(images, nbins=25):
    """Same 25 radial bins/FFT normalization as evaluate_late_start; DC not removed."""
    x = images[:, 0].astype(np.float64)
    fourier = np.fft.fft2(x)
    power = np.abs(fourier) ** 2 / (128 * 128)
    frequency = np.fft.fftfreq(128) * 128
    radius = np.hypot(frequency[:, None], frequency[None, :])
    edges = np.linspace(0, radius.max(), nbins + 1)
    index = np.clip(np.digitize(radius, edges) - 1, 0, nbins - 1)
    counts = np.bincount(index.ravel(), minlength=nbins)
    sums = np.stack([np.bincount(index.ravel(), weights=p.ravel(), minlength=nbins) for p in power])
    return sums / counts, (edges[:-1] + edges[1:]) / 2


def groups(cosine):
    return {"near-copy": cosine > .98,
            "intermediate": (cosine >= .8) & (cosine <= .98),
            "low-similarity": cosine < .8}


def training_progress(root, arm_index):
    candidates = sorted((Path(root) / "logs").glob(f"train_*_{arm_index}.out"),
                        key=lambda p: p.stat().st_mtime)
    if not candidates:
        return {"completed_epoch": None, "nominal_updates": None, "log": None}
    path = candidates[-1]
    with path.open("rb") as handle:
        handle.seek(max(0, path.stat().st_size - 256_000))
        text = handle.read().decode(errors="replace")
    epochs = re.findall(r"Epoch (\d+) — avg loss:", text)
    epoch = int(epochs[-1]) if epochs else None
    return {"completed_epoch": epoch, "nominal_updates": (epoch + 1) * 32 if epoch is not None else None,
            "log": str(path)}


def load_context(root, expected_plan_sha=PLAN_SHA):
    root = Path(root)
    path = root / "plan/plan.json"
    if not path.is_file():
        return None
    if file_sha(path) != expected_plan_sha:
        raise ValueError("A40 plan hash changed; do not silently read a different experiment")
    plan = read_json(path)
    if (plan.get("status") != "prepared" or plan.get("code_revision") != TRAIN_REVISION
            or plan.get("matrix_plan_sha256") != MATRIX_SHA
            or [(r["name"], r["patch_size"], r["initialization"]) for r in plan["arms"]]
            != [("p8_zero", 8, "zero"), ("p4_native", 4, "native"), ("p4_zero", 4, "zero")]):
        raise ValueError("Unexpected experiment identity or arm coverage")
    for arm in plan["arms"]:
        if file_sha(arm["config"]) != arm["config_sha256"]:
            raise ValueError(f"Frozen YAML changed: {arm['name']}")
    config = yaml.safe_load(Path(plan["arms"][0]["config"]).read_text())
    reference, metadata = load_native_training_reference(config)
    fields(reference, 256)
    if metadata != plan["training_data_reference"] or array_sha(reference) != plan["training_reference_sha256"]:
        raise ValueError("Retained-slice data/normalization differs from the frozen training reference")
    noise_path = root / "plan/initial_noise.npz"
    if file_sha(noise_path) != plan["noise_file_sha256"]:
        raise ValueError("Frozen initial-noise file changed")
    with np.load(noise_path, allow_pickle=False) as data:
        noise = fields(data["initial_noise"].copy(), 128)
    if array_sha(noise) != plan["noise_batch_sha256"]:
        raise ValueError("Frozen noise fields changed")
    reference_power, k = radial_power(reference)
    return {"root": root, "plan": plan, "plan_sha": expected_plan_sha, "reference": reference,
            "noise": noise, "reference_power": reference_power, "k": k,
            "real_boundary": {p: boundary_ratio(reference, p) for p in (4, 8)}}


def checked_training_record(context, arm):
    path = Path(arm["checkpoint_dir"]) / "complete_record/complete.json"
    if not path.is_file():
        return None
    report = read_json(path)
    if (report.get("status") != "complete" or report.get("arm") != arm
            or report.get("plan_sha256") != context["plan_sha"]
            or report.get("nominal_optimizer_steps") != 300000
            or not 0 < report.get("successful_optimizer_steps", 0) <= 300000):
        raise ValueError(f"Invalid training completion receipt: {arm['name']}")
    return report


def load_arm(context, arm):
    """Return None only for pending outputs. Reject malformed/completed-but-missing results."""
    folder = context["root"] / "samples" / arm["name"]
    receipt = folder / "complete.json"
    if not receipt.is_file():
        return None
    training = checked_training_record(context, arm)
    report = read_json(receipt)
    path = folder / "samples.npz"
    if (training is None or report.get("status") != "complete" or report.get("arm") != arm
            or report.get("plan_sha256") != context["plan_sha"] or report.get("n") != 128
            or report.get("noise_sha256") != context["plan"]["noise_batch_sha256"]
            or not path.is_file() or file_sha(path) != report.get("samples_sha256")):
        raise ValueError(f"Invalid completed sample receipt: {arm['name']}")
    with np.load(path, allow_pickle=False) as data:
        samples = fields(data["samples"].copy(), 128)
        if not np.array_equal(data["initial_noise"], context["noise"]):
            raise ValueError("Initial noises are not byte-paired")
        expected = {"config_sha256": arm["config_sha256"], "checkpoint": arm["expected_checkpoint"],
                    "reference_sha256": context["plan"]["training_reference_sha256"],
                    "data_reference_contract": CONTRACT,
                    "initial_noise_batch_sha256": context["plan"]["noise_batch_sha256"],
                    "raw_weights": True, "model_batch_size": 2, "seed": 123, "step_seed": 124, "num_steps": 50}
        if any(scalar(data, key) != value for key, value in expected.items()):
            raise ValueError("Sample provenance differs from the reviewed A40 protocol")
        scheduler = json.loads(scalar(data, "scheduler_config_json"))
        if any(scheduler.get(key) != value for key, value in {
                "prediction_type": "v_prediction", "algorithm_type": "dpmsolver++", "solver_order": 2}.items()):
            raise ValueError("Sample scheduler differs from raw DPM++ order-2 control")
        stored_cosine = data["max_cos"].copy()
    result = analyze(samples, context, arm["name"], arm["patch_size"], path)
    if stored_cosine.shape != (128,) or not np.allclose(stored_cosine, result["cosine"], atol=2e-5, rtol=0):
        raise ValueError("Recomputed cosine differs from saved diagnostic")
    result["training"] = training
    return result


def load_baselines(context):
    matrix_path = Path(context["plan"]["matrix_plan_path"])
    if file_sha(matrix_path) != MATRIX_SHA:
        raise ValueError("Original matrix plan changed")
    matrix = read_json(matrix_path)
    results, status = {}, []
    for name in ("dit_l16_fresh300k", "dit_l16_seed456_500k", "dit_l8_200k"):
        folder = matrix_path.parents[1] / "tasks" / f"{name}__dpm50"
        path, receipt = folder / "samples.npz", folder / "complete.json"
        if not receipt.is_file():
            status.append({"condition": LABELS[name], "sample_state": "not available", "detail": str(folder)})
            continue
        report = read_json(receipt)
        model = next(r for r in matrix["models"] if r["name"] == name)
        if (report.get("status") != "complete" or report.get("plan_sha256") != MATRIX_SHA
                or report.get("task") != {"model": name, "condition": "dpm50"}
                or report.get("reference_sha256") != context["plan"]["reference_sha256"]
                or not path.is_file() or file_sha(path) != report.get("artifacts_sha256", {}).get("samples.npz")
                or file_sha(model["config"]) != model["config_sha256"]):
            raise ValueError(f"Original matrix output changed: {name}")
        config = yaml.safe_load(Path(model["config"]).read_text())
        reference, _ = load_native_training_reference(config)
        if array_sha(reference) != context["plan"]["training_reference_sha256"]:
            raise ValueError(f"Baseline does not use the same retained training subset: {name}")
        with np.load(path, allow_pickle=False) as data:
            samples = fields(data["samples"].copy(), 128)
            if (scalar(data, "matrix_condition") != "dpm50" or scalar(data, "num_steps") != 50
                    or scalar(data, "ema_sigma_rel") != -1.0
                    or scalar(data, "config_sha256") != model["config_sha256"]
                    or scalar(data, "initial_noise_batch_sha256") != context["plan"]["noise_batch_sha256"]
                    or not np.array_equal(data["initial_noise"], context["noise"])):
                raise ValueError(f"Baseline sampler/noise pairing changed: {name}")
        results[name] = analyze(samples, context, name, 8, path)
        status.append({"condition": LABELS[name], "sample_state": "ready (re-scored)", "detail": str(path)})
    return results, status


def analyze(samples, context, name, patch, path):
    cosine, nearest = nearest_training(samples, context["reference"])
    power, k = radial_power(samples)
    if not np.array_equal(k, context["k"]):
        raise ValueError("Inconsistent Fourier bins")
    return {"name": name, "label": LABELS[name], "patch": patch, "path": str(path), "samples": samples,
            "cosine": cosine, "nearest": nearest, "groups": groups(cosine), "power": power,
            "boundary": {p: boundary_ratio(samples, p) for p in (4, 8)}}


def read_results(root, expected_plan_sha=PLAN_SHA, include_baselines=True):
    context = load_context(root, expected_plan_sha)
    status, results = [], {}
    if context is None:
        for name in ("p8_zero", "p4_native", "p4_zero"):
            status.append({"condition": LABELS[name], "training_state": "unknown (plan unavailable)",
                           "sample_state": "pending / path unavailable", "nominal_updates": None})
        return None, results, status
    for i, arm in enumerate(context["plan"]["arms"]):
        progress = training_progress(root, i)
        try:
            training = checked_training_record(context, arm)
            result = load_arm(context, arm)
            if result is not None:
                results[arm["name"]] = result
            status.append({"condition": LABELS[arm["name"]],
                           "training_state": "validated complete" if training else "not yet validated complete",
                           "sample_state": "ready" if result else "pending (auto sampler waits for ALL arms)",
                           "nominal_updates": 300000 if training else progress["nominal_updates"],
                           "successful_updates": training["successful_optimizer_steps"] if training else None,
                           "detail": progress["log"]})
        except (ValueError, KeyError, OSError) as error:
            status.append({"condition": LABELS[arm["name"]], "sample_state": "ERROR — excluded from comparison",
                           "training_state": "inspect receipt/log", "detail": str(error)})
    if include_baselines:
        baselines, baseline_status = load_baselines(context)
        results = {**baselines, **results}
        status += baseline_status
    return context, results, status


def summary_rows(context, results):
    rows = []
    real_mean = context["reference_power"].mean(axis=0)
    high_k = (context["k"] >= 32) & (context["k"] <= 64)
    for result in results.values():
        for group, mask in result["groups"].items():
            row = {"condition": result["label"], "population": group, "count": int(mask.sum()),
                   "fraction (%)": 100 * float(mask.mean()), "total draws": len(mask)}
            for patch in (4, 8):
                real = context["real_boundary"][patch]
                row[f"B{patch} / real median"] = (float(np.nanmedian(result["boundary"][patch][mask]) / np.nanmedian(real))
                                                   if mask.any() else np.nan)
                row[f"B{patch} above real p99 (%)"] = (100 * float(np.mean(result["boundary"][patch][mask] > np.nanquantile(real, .99)))
                                                       if mask.any() else np.nan)
            row["high-k mean power ratio"] = (float((result["power"][mask].mean(axis=0) / real_mean)[high_k].mean())
                                               if mask.any() else np.nan)
            rows.append(row)
    return rows


def show_contact_sheet(result, context, page=0, per_page=16, population="all"):
    import matplotlib.pyplot as plt
    if population != "all" and population not in result["groups"]:
        raise ValueError(f"Choose all or one of {GROUPS}")
    indices = (np.arange(len(result["samples"])) if population == "all"
               else np.flatnonzero(result["groups"][population]))
    if not len(indices):
        print(f"{result['label']}: no {population} maps. Nothing to display.")
        return
    start, stop = page * per_page, min((page + 1) * per_page, len(indices))
    if not 0 <= start < stop:
        raise ValueError(f"Choose page 0..{(len(indices) - 1) // per_page} for {population}")
    limits = np.quantile(context["reference"], [.001, .999])
    fig, axes = plt.subplots(4, 4, figsize=(12, 12), layout="constrained")
    for axis in axes.flat:
        axis.axis("off")
    for index, axis in zip(indices[start:stop], axes.flat):
        image = axis.imshow(result["samples"][index, 0], cmap="viridis", vmin=limits[0], vmax=limits[1], interpolation="nearest")
        axis.set_title(f"draw {index} · cosine {result['cosine'][index]:.3f}\nB{result['patch']}={result['boundary'][result['patch']][index]:.2f}", fontsize=10)
    fig.suptitle(f"{result['label']}\n{population}: page {page}; {len(indices)} maps total; fixed real-reference color range", fontsize=13)
    fig.colorbar(image, ax=axes.ravel().tolist(), shrink=.65, label="Normalized pixel value (color range clips reference tails)")
    plt.show()


def inspect_draw(result, context, index):
    import matplotlib.pyplot as plt
    x = result["samples"][index, 0]
    nearest = int(result["nearest"][index])
    real_map = context["reference"][nearest, 0]
    patch = result["patch"]
    limits = np.quantile(context["reference"], [.001, .999])
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.6), layout="constrained")
    for axis, value, title in zip(axes, (x, x, real_map),
                                 (f"Generated draw {index}", f"Central zoom, model grid {patch} px", f"Nearest training map {nearest}")):
        image = axis.imshow(value, vmin=limits[0], vmax=limits[1], cmap="viridis", interpolation="nearest")
        axis.set_title(title)
        axis.set_xticks([]); axis.set_yticks([])
    axes[1].set_xlim(31.5, 95.5); axes[1].set_ylim(95.5, 31.5)
    for boundary in range(32, 97, patch):
        axes[1].axvline(boundary - .5, color="white", alpha=.5, linewidth=.55)
        axes[1].axhline(boundary - .5, color="white", alpha=.5, linewidth=.55)
    fig.colorbar(image, ax=axes.tolist(), shrink=.8, label="Normalized pixel value")
    fig.suptitle(f"{result['label']} · max cosine={result['cosine'][index]:.4f}")
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
    pixel_edges = np.linspace(float(context["reference"].min()), float(context["reference"].max()), 100)
    axes[0].hist(context["reference"].ravel(), bins=pixel_edges, density=True, histtype="step", color="#333333", label="All 256 real maps")
    axes[0].hist(x.ravel(), bins=pixel_edges, density=True, histtype="step", color="#c05c33", label=f"Generated draw {index}")
    axes[0].set(xlabel="Normalized pixel value", ylabel="Probability density", title="One-map pixel distribution")
    axes[0].legend(fontsize=9)
    k = context["k"]
    valid = (k > 4) & (k <= 64)
    mean = context["reference_power"].mean(axis=0)
    lo, hi = np.quantile(context["reference_power"] / mean, [.1, .9], axis=0)
    axes[1].fill_between(k[valid], lo[valid], hi[valid], color="#dddddd", label="Real maps: p10–p90 (not a CI)")
    axes[1].plot(k[valid], result["power"][index, valid] / mean[valid], color="#c05c33", label="Generated draw")
    axes[1].plot(k[valid], context["reference_power"][nearest, valid] / mean[valid], color="#31688e", linestyle="--", label="Nearest real map")
    axes[1].axhline(1, color="#333333", linestyle=":")
    axes[1].set(xlabel="k (Fourier grid units)", ylabel="Per-map power / mean real power", title="One-map power-spectrum ratio")
    axes[1].legend(fontsize=8)
    def jump_profile(value):
        return (np.abs(np.diff(value, axis=-1)).mean(axis=-2) + np.abs(np.diff(value, axis=-2)).mean(axis=-1)) / 2
    profiles = jump_profile(context["reference"][:, 0])
    lo, hi = np.quantile(profiles, [.1, .9], axis=0)
    positions = np.arange(1, 128)
    axes[2].fill_between(positions, lo, hi, color="#dddddd", label="Real maps: p10–p90")
    axes[2].plot(positions, jump_profile(x), color="#c05c33", label="Generated draw")
    axes[2].plot(positions, profiles.mean(axis=0), color="#333333", linewidth=1, label="Real mean")
    for boundary in range(patch, 128, patch):
        axes[2].axvline(boundary, color="#31688e", alpha=.18, linewidth=.7)
    axes[2].set(xlabel="Pixel edge position (row/column averaged)", ylabel="Mean absolute neighboring-pixel jump", title=f"Jump profile; guides every {patch} pixels")
    axes[2].legend(fontsize=8)
    plt.show()
    print(f"B4={result['boundary'][4][index]:.3f}; B8={result['boundary'][8][index]:.3f}. Compare against real-map distributions, not an assumed value of 1.")
    print(f"Outside histogram range: {100*np.mean((x < pixel_edges[0]) | (x > pixel_edges[-1])):.3f}% of generated pixels.")


In [ ]:
context, results, readiness_rows = read_results(ABLATION_ROOT, EXPECTED_PLAN_SHA, INCLUDE_OLD_CONTROLS)
readiness = pd.DataFrame(readiness_rows)
display(readiness)
if context is None:
    display(Markdown('**Plan/data path unavailable.** On Great Lakes, check `ABLATION_ROOT`; no results are assumed.'))
else:
    print('Training reference SHA256:', array_sha(context['reference']))
    print('Initial noise SHA256:', array_sha(context['noise']))
    print('Ready sample sets:', ', '.join(results) or 'none yet')
    print('Missing sample sets are pending; no result is imputed.')

## Results

### 3. Population comparison — similarity is NOT validity

**Near-copy:** max training cosine > 0.98. **Intermediate:** 0.80–0.98 inclusive. **Low-similarity:** < 0.80. These labels say whether a generated map resembles a training map; they do not say whether it is physically valid.

`B4` / `B8` = mean absolute neighboring-pixel jump across 4- / 8-pixel grid boundaries, divided by the mean jump elsewhere. Tables divide that ratio by the **real-map median**. Elevated values indicate grid-aligned discontinuities. The fraction above the real p99 is a diagnostic, not a validated failure classifier. Empty populations have missing diagnostics, not zero error.

In [ ]:
summary = pd.DataFrame(summary_rows(context, results)) if context is not None else pd.DataFrame()
if summary.empty:
    print('No completed samples to score yet.')
else:
    display(summary.round(3))
    fig, axis = plt.subplots(figsize=(12, 4.8), layout='constrained')
    labels = list(results)
    left = np.zeros(len(labels))
    for group in GROUPS:
        counts = np.array([int(results[name]['groups'][group].sum()) for name in labels])
        percent = counts / 128 * 100
        axis.barh(np.arange(len(labels)), percent, left=left, color=COLORS[group], label=group)
        for row, (width, offset, count) in enumerate(zip(percent, left, counts)):
            if width >= 8:
                axis.text(offset + width/2, row, f'{count}/128', ha='center', va='center', color='white', fontsize=9)
        left += percent
    axis.set_yticks(np.arange(len(labels)), [results[name]['label'] for name in labels])
    axis.invert_yaxis()
    axis.set(xlim=(0, 100), xlabel='Fraction of 128 paired generated maps (%)', title='Saved samples grouped by similarity — not by validity')
    axis.legend(loc='upper center', bbox_to_anchor=(.5, 1.22), ncol=3, fontsize=9)
    plt.show()

### 4. Patch-boundary distributions

Each dot is one generated map. Y = its boundary-jump ratio divided by the real-map median. A value of 2 means twice the real median grid discontinuity. Dashed lines show the real-map p99 on that same scale. Both pixel grids are shown, including the old 8-pixel grid for patch-4 models.

In [ ]:
if context is not None and results:
    fig, axes = plt.subplots(len(results), 2, figsize=(14, 3.4*len(results)), squeeze=False, layout='constrained')
    for row, result in enumerate(results.values()):
        for column, patch in enumerate((4, 8)):
            axis = axes[row, column]
            real = context['real_boundary'][patch]
            real_median = np.nanmedian(real)
            axis.axhline(1, color='#333333', linestyle=':', label='Real median')
            axis.axhline(np.nanquantile(real, .99)/real_median, color='#333333', linestyle='--', label='Real p99')
            for group, mask in result['groups'].items():
                if mask.any():
                    axis.scatter(result['cosine'][mask], result['boundary'][patch][mask]/real_median,
                                 color=COLORS[group], s=17, alpha=.6, label=group)
            axis.set(xlabel='Maximum pixel cosine to any training map', ylabel=f'B{patch} / real-map median B{patch}',
                     title=f'{result["label"]} · {patch}-pixel grid')
            axis.legend(fontsize=8, ncol=2)
    top = max(axis.get_ylim()[1] for axis in axes.flat)
    low = min(axis.get_ylim()[0] for axis in axes.flat)
    for axis in axes.flat:
        axis.set_ylim(low, top)
    plt.show()
else:
    print('Patch-boundary comparison pending.')

### 5. Power spectra — split copies from non-copies

**X axis:** spatial frequency `k` in Fourier grid units; larger k = smaller-scale structure. **Y axis:** population mean power / mean power of all 256 real training maps. 1 means matching mean power at that scale; 2 means twice the power. Mean ratios are computed per frequency, not averaged with the copy population.

Gray bands show the real maps' p10–p90 **map-to-map spread, not a confidence interval**. These are spectra of the **normalized pixel fields**, not physical density-contrast cosmological P(k). The first bin includes DC and is omitted here; plotted bin centers satisfy 4 < k ≤ 64 (axial Nyquist). An averaged spectrum near 1 alone does not establish image validity.

In [ ]:
if context is not None and results:
    real_mean = context['reference_power'].mean(axis=0)
    lo, hi = np.quantile(context['reference_power']/real_mean, [.1, .9], axis=0)
    k = context['k']; valid = (k > 4) & (k <= 64)
    fig, axes = plt.subplots(len(results), 1, figsize=(10, 3.3*len(results)), squeeze=False, layout='constrained')
    for axis, result in zip(axes[:, 0], results.values()):
        axis.fill_between(k[valid], lo[valid], hi[valid], color='#dddddd', label='Real-map p10–p90 spread')
        axis.axhline(1, color='#333333', linestyle=':')
        for group, mask in result['groups'].items():
            if mask.any():
                ratio = result['power'][mask].mean(axis=0)/real_mean
                axis.plot(k[valid], ratio[valid], color=COLORS[group], marker='.', label=f'{group}: n={mask.sum()}')
        axis.set(xlabel='k (Fourier grid units)', ylabel='Mean generated / mean real power', title=result['label'])
        axis.legend(fontsize=9, ncol=2)
    # Same y-range for every condition; avoid hiding a failed tail with a fixed cutoff.
    top = max(axis.get_ylim()[1] for axis in axes[:, 0])
    for axis in axes[:, 0]:
        axis.set_ylim(0, top)
    plt.show()
else:
    print('Power-spectrum comparison pending.')

### 6. Compare identical noise draws across conditions

Rows use the **same initial noise**. The six columns are evenly spaced indices, not hand-picked successes. Every map uses the same color range fitted to the real reference; extreme tails can clip visually. The titles expose draw index and training cosine.

In [ ]:
PAIRED_INDICES = [0, 25, 50, 76, 101, 127]
if context is not None and results:
    limits = np.quantile(context['reference'], [.001, .999])
    fig, axes = plt.subplots(len(results), len(PAIRED_INDICES), figsize=(17, 2.9*len(results)), squeeze=False, layout='constrained')
    for row, result in enumerate(results.values()):
        for column, index in enumerate(PAIRED_INDICES):
            axis = axes[row, column]
            image = axis.imshow(result['samples'][index, 0], vmin=limits[0], vmax=limits[1], cmap='viridis', interpolation='nearest')
            axis.set_title(f'draw {index} · cos {result["cosine"][index]:.3f}', fontsize=10)
            axis.set_xticks([]); axis.set_yticks([])
            if column == 0:
                axis.set_ylabel(result['label'].replace(': ', ':\n').replace(' + ', '\n+ '), fontsize=9)
    fig.colorbar(image, ax=axes.ravel().tolist(), shrink=.65, label='Normalized pixel value')
    plt.show()
else:
    print('Paired image comparison pending.')

### 7. Browse every generated map

Set `GALLERY_MODEL` to a ready name: `p8_zero`, `p4_native`, `p4_zero`, `dit_l16_fresh300k`, `dit_l16_seed456_500k`, or `dit_l8_200k`. Page 0–7 shows 16 maps at a time. Set `SHOW_ALL_PAGES = True` to display **every image** for that condition. Set `GALLERY_POPULATION = 'low-similarity'` to isolate those maps, or leave `'all'` to see all 128. Original draw indices are preserved. Default picks the first ready new arm, or the old L16 baseline while new results are pending.

In [ ]:
GALLERY_MODEL = next((name for name in ('p8_zero', 'p4_native', 'p4_zero', 'dit_l16_fresh300k') if name in results), None)
GALLERY_PAGE = 0
GALLERY_POPULATION = 'all'  # 'all', 'near-copy', 'intermediate', or 'low-similarity'
SHOW_ALL_PAGES = False

if GALLERY_MODEL in results:
    result = results[GALLERY_MODEL]
    gallery_count = len(result['samples']) if GALLERY_POPULATION == 'all' else int(result['groups'][GALLERY_POPULATION].sum())
    pages = range((gallery_count + 15)//16) if SHOW_ALL_PAGES else [GALLERY_PAGE]
    for page in pages:
        show_contact_sheet(result, context, page, population=GALLERY_POPULATION)
    if not gallery_count:
        print('No maps in the selected population.')
else:
    print('Selected gallery result is not ready. Ready names:', list(results))

### 8. Inspect one map: pixels, spectrum, and boundaries

Choose any `INSPECT_MODEL` and draw index 0–127. Default selects the **lowest-cosine** map from the chosen condition (deliberately an extreme case, not a representative sample). First figure: full map, zoom with its actual pixel-patch grid, nearest real map. Second figure: pixel PDF, individual P(k) ratio, and the actual pixel-jump profile. This is where you decide whether non-copies are plausible new maps or still blocky artifacts.

In [ ]:
INSPECT_MODEL = GALLERY_MODEL
INSPECT_INDEX = int(results[INSPECT_MODEL]['cosine'].argmin()) if INSPECT_MODEL in results else 0

if INSPECT_MODEL in results:
    inspect_draw(results[INSPECT_MODEL], context, INSPECT_INDEX)
else:
    print('Individual inspection pending.')

### 9. Pair the inspected draw across all ready conditions

This table compares the same draw index, not independently selected worst maps. `B4` and `B8` here are the raw edge/inside ratios. A fix should improve image/spectrum/boundary agreement, not merely turn more maps into training copies.

In [ ]:
if context is not None and results:
    paired = []
    for result in results.values():
        paired.append({'condition': result['label'], 'draw': INSPECT_INDEX,
                       'max training cosine': float(result['cosine'][INSPECT_INDEX]),
                       'B4': float(result['boundary'][4][INSPECT_INDEX]),
                       'B8': float(result['boundary'][8][INSPECT_INDEX])})
    display(pd.DataFrame(paired).round(4))

## Takeaways

Do not equate a low-copy fraction with a failed generator, or a high-copy fraction with good generalization.

- **Initialization candidate helps:** `p8_zero` improves map quality and real-calibrated boundary/spectrum agreement relative to old patch-8 L16.
- **Patch-size candidate helps:** `p4_native` improves those diagnostics while remaining native-initialized.
- **Combined candidate helps:** `p4_zero` is better than either single change; inspect whether copies/non-copies differ.
- **Not settled:** missing samples, visibly blocky non-copies, or improvements only in the mixed-population average.

Successful training/lower loss alone does not answer this question. These are single-seed, N=256 intervention tests; they do not establish universal causality. This notebook never launches an early sampling job: use the existing automatic sampler, or obtain a separately approved run if early results are needed.